# Breast Cancer Classification with Machine Learning

This notebook builds and compares several classifiers on the **Breast Cancer Wisconsin** dataset (bundled with scikit-learn) to predict whether a tumor is malignant or benign.

**Steps:** load data → explore → preprocess → train multiple models → evaluate → tune the best → feature importance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load the data

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print("Shape:", X.shape)
print("Classes:", dict(zip(data.target_names, np.bincount(y))))
X.head()

## 2. Exploratory data analysis

In [ ]:
print("Missing values:", X.isna().sum().sum())
X.describe().T.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class balance
pd.Series(y).map(dict(enumerate(data.target_names))).value_counts().plot.bar(
    ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Class distribution")
axes[0].set_ylabel("Count")

# Correlation of features with target
corr = X.corrwith(y).sort_values()
corr.plot.barh(ax=axes[1], figsize=(13, 8), color="#55A868")
axes[1].set_title("Feature correlation with target (1 = benign)")

plt.tight_layout()
plt.show()

## 3. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 4. Compare models with cross-validation

Scaling is wrapped in a `Pipeline` so it is fit only on training folds (no data leakage).

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
    "SVM (RBF)": SVC(probability=True, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    results[name] = scores
    print(f"{name:20s} CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

plt.figure(figsize=(8, 4))
plt.boxplot(results.values(), labels=results.keys())
plt.xticks(rotation=20)
plt.ylabel("CV accuracy")
plt.title("Model comparison (5-fold CV)")
plt.show()

## 5. Hyperparameter tuning

We tune an SVM (typically one of the strongest on this dataset) using grid search.

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))])

param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.01, 0.001],
    "clf__kernel": ["rbf", "linear"],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.4f}")

## 6. Final evaluation on the held-out test set

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}\n")
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=data.target_names).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("ROC curve")

plt.tight_layout()
plt.show()

## 7. Feature importance

A Random Forest gives a quick view of which measurements matter most.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index, color="#4C72B0")
plt.title("Top 10 most important features")
plt.xlabel("Importance")
plt.show()

## Summary

- Multiple classifiers were compared with cross-validation on a scaled pipeline.
- The tuned SVM reached high test accuracy and ROC-AUC on unseen data.
- Size and shape features such as *worst perimeter*, *worst radius* and *worst concave points* drive predictions most.

**Next ideas:** try `XGBoost`/`LightGBM`, use SHAP for explainability, or tune the decision threshold to favor recall on the malignant class.